# Gene Coordinate Annotation


## Description

We use a gene coordinate annotation pipeline based on [`pyqtl`, as demonstrated here](https://github.com/broadinstitute/gtex-pipeline/blob/master/qtl/src/eqtl_prepare_expression.py). This adds genomic coordinate annotations to gene-level molecular phenotype files generated in `gct` format and converts them to `bed` format for downstreams analysis.


### Alternative implementation

Previously we use `biomaRt` package in R instead of code from `pyqtl`. The core function calls are:

```r
    ensembl = useEnsembl(biomart = "ensembl", dataset = "hsapiens_gene_ensembl", version = "$[ensembl_version]")
    ensembl_df <- getBM(attributes=c("ensembl_gene_id","chromosome_name", "start_position", "end_position"),mart=ensembl)
```

We require ENSEMBL version to be specified explicitly in this pipeline. As of 2021 for the Brain xQTL project, we use ENSEMBL version 103.

## Input Files

| File | Description |
|------|-------------|
| `input/rnaseq/protocol_example.rnaseq.bed.gz` | Toy bulk RNA-seq expression matrix (gene-level). |
| `input/proteomics/protocol_example.protein.no_coord.tsv` | Toy protein matrix with `gene_id\|UniProt` IDs, no coordinates. |
| `input/rnaseq/protocol_example.rnaseq.gene_ID.tsv` | Toy expression matrix with a `gene_ID` column, for the biomaRt example. |
| `input/rnaseq/protocol_example.leafcutter.intron_count.tsv` | Toy leafcutter intron-count table (IDs `chr:start:end:clu_N_strand`). |
| `input/rnaseq/protocol_example.leafcutter.phenotype.bed.gz` | Toy leafcutter intron-excision phenotype matrix. |
| `input/rnaseq/protocol_example.psichomics.phenotype.tsv` | Toy psichomics phenotype matrix (IDs end in `_<gene_id>`). |
| `reference_data/Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.ERCC.gtf` | Collapsed gene-model GTF (gene/protein annotation). |
| `reference_data/Homo_sapiens.GRCh38.103.chr.gtf` | Full gene-model GTF with exons (leafcutter/psichomics). |


## Output Files

| Output | Description |
|--------|-------------|
| Coordinate-annotated phenotype (`*.bed.gz`) | The input molecular phenotype matrix with genomic coordinates (`#chr start end ID`) prepended, ready for downstream QTL analysis. |
| Cluster-to-gene map (`*.leafcutter.clusters_to_genes.txt`) | (leafcutter) mapping of each intron cluster to its gene(s). |
| Phenotype group file (`*.phenotype_group.txt`) | (leafcutter/psichomics) mapping of each feature ID to its gene, for grouped QTL analysis. |


## Steps

The example below adds genomic coordinates to a molecular phenotype matrix using the collapsed gene-model GTF. We show two cases: gene expression and proteins.


### Gene Expression

Timing: <1 min


Annotate a gene-level expression matrix. Each feature ID is an ENSEMBL gene ID that is matched directly against the GTF to obtain its chromosome, start and end coordinates.

In [ ]:
sos run pipeline/gene_annotation.ipynb annotate_coord \
    --cwd output/gene_annotation \
    --phenoFile input/rnaseq/protocol_example.rnaseq.bed.gz \
    --coordinate-annotation reference_data/Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.ERCC.gtf \
    --phenotype-id-column gene_id


The output `protocol_example.rnaseq.bed.gz` is written to `output/gene_annotation/`, with genomic coordinates (`#chr start end ID`) prepended to the expression matrix.


### Proteins

Timing: <1 min


Annotate a protein matrix. Here each feature ID has the form `gene_id\|UniProt`; the gene ID portion is matched against the GTF for coordinates and the UniProt portion is retained in the output feature ID. Use `--molecular-trait-type protein` to select this behaviour.

In [ ]:
sos run pipeline/gene_annotation.ipynb annotate_coord \
    --cwd output/gene_annotation \
    --phenoFile input/proteomics/protocol_example.protein.no_coord.tsv \
    --coordinate-annotation reference_data/Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.ERCC.gtf \
    --phenotype-id-column gene_id \
    --molecular-trait-type protein


### Leafcutter clusters to genes

Timing: <2 min


For leafcutter splicing data, intron clusters first need to be assigned to genes. The input is a leafcutter intron-count table whose feature IDs have the form `chr:start:end:clu_N_strand`; `--coordinate-annotation` is a full gene-model GTF (with exon records). The default `--map-stra site` maps introns to genes by matching splice-site coordinates to exon boundaries.

In [ ]:
sos run pipeline/gene_annotation.ipynb map_leafcutter_cluster_to_gene \
    --cwd output/gene_annotation \
    --phenoFile input/rnaseq/protocol_example.leafcutter.phenotype.bed.gz \
    --intron-count input/rnaseq/protocol_example.leafcutter.intron_count.tsv \
    --coordinate-annotation reference_data/Homo_sapiens.GRCh38.103.chr.gtf \
    --map-stra site


### Leafcutter isoforms

Timing: <2 min


This example turns the raw leafcutter intron-excision output into a phenotype matrix that TensorQTL can use. Building on the cluster-to-gene mapping from the previous example, it attaches genomic coordinates and a gene assignment to every intron, then writes a coordinate-annotated phenotype BED together with a phenotype-group file that links each intron back to its gene. The same `--intron-count` table and `--coordinate-annotation` GTF used above are supplied here.

Along the way the step produces three intermediate files, shown below with a few example rows from the toy data.

**1. Exon list** — exon records parsed from the annotation GTF, used to match intron splice sites to genes (`<intron_count>.exon_list`):

| chr | start | end | strand | gene_id | gene_name |
|-----|-------|-----|--------|---------|-----------|
| 1 | 111869 | 112227 | + | ENSG00000223972 | DDX11L1 |
| 1 | 112613 | 112721 | + | ENSG00000223972 | DDX11L1 |

**2. Clusters-to-genes** — each leafcutter cluster mapped to the gene it belongs to (`<intron_count>.leafcutter.clusters_to_genes.txt`):

| clu | genes |
|-----|-------|
| 22:clu_10_+ | ENSG00000236052 |
| 22:clu_11_- | ENSG00000185264 |

**3. Phenotype group** — links each annotated intron feature to its gene, so introns from the same gene are grouped together (`<phenotype>.phenotype_group.txt`):

| ID | gene |
|----|------|
| 22:19149095:19149663:clu_14_-:ENSG00000063515 | ENSG00000063515 |
| 22:19149916:19150025:clu_14_-:ENSG00000063515 | ENSG00000063515 |

In [ ]:
sos run pipeline/gene_annotation.ipynb annotate_leafcutter_isoforms \
    --cwd output/gene_annotation \
    --phenoFile input/rnaseq/protocol_example.leafcutter.phenotype.bed.gz \
    --intron-count input/rnaseq/protocol_example.leafcutter.intron_count.tsv \
    --coordinate-annotation reference_data/Homo_sapiens.GRCh38.103.chr.gtf \
    --map-stra site


### Psichomics isoforms

Timing: <2 min


For psichomics splicing quantifications, each event ID ends in `_<gene_id>`; the gene ID is matched against the GTF to obtain coordinates. The output is a coordinate-annotated phenotype BED plus a phenotype-group file.

In [ ]:
sos run pipeline/gene_annotation.ipynb annotate_psichomics_isoforms \
    --cwd output/gene_annotation \
    --phenoFile input/rnaseq/protocol_example.psichomics.phenotype.tsv \
    --coordinate-annotation reference_data/Homo_sapiens.GRCh38.103.chr.gtf


### Alternative coordinate annotation with biomaRt

Timing: <2 min (depends on network)


Instead of a local GTF, coordinates can be fetched from the Ensembl biomaRt web service. The phenotype matrix must have a `gene_ID` column of ENSEMBL gene IDs. `--ensembl-version` selects the Ensembl release; use a release whose server is reachable from your network (e.g. a current release). This step requires internet access to Ensembl.

In [ ]:
sos run pipeline/gene_annotation.ipynb annotate_coord_biomart \
    --cwd output/gene_annotation \
    --phenoFile input/rnaseq/protocol_example.rnaseq.gene_ID.tsv \
    --ensembl-version 115


## Command Interface

In [2]:
sos run gene_annotation.ipynb -h

usage: sos run gene_annotation.ipynb
               [workflow_name | -t targets] [options] [workflow_options]
  workflow_name:        Single or combined workflows defined in this script
  targets:              One or more targets to generate
  options:              Single-hyphen sos parameters (see "sos run -h" for details)
  workflow_options:     Double-hyphen workflow-specific parameters

Workflows:
  region_list_generation
  annotate_coord
  annotate_coord_biomart
  map_leafcutter_cluster_to_gene
  annotate_leafcutter_isoforms
  annotate_psichomics_isoforms

Global Workflow Options:
  --cwd output (as path)
                        Work directory & output directory
  --phenoFile VAL (as path, required)
                        Molecular phenotype matrix
  --job-size 1 (as int)
                        For cluster jobs, number commands to run per job
  --walltime 5h
                        Wall clock time expected
  --mem 16G
                        Memory expected
  --numThreads 1 (as 

## Setup and global parameters

In [ ]:
[global]
# Work directory & output directory
parameter: cwd = path("output")
# Molecular phenotype matrix
parameter: phenoFile = path
# For cluster jobs, number commands to run per job
parameter: job_size = 1
# Wall clock time expected
parameter: walltime = "5h"
# Memory expected
parameter: mem = "16G"
# Number of threads
parameter: numThreads = 1
parameter: container = ""


## Implementation using `pyqtl`

Implementation based on [GTEx pipeline](https://github.com/broadinstitute/gtex-pipeline/blob/master/qtl/src/eqtl_prepare_expression.py).

Following step serves to annotate coord for gene expression file.

In [ ]:
[annotate_coord]
# A file to map sample ID from expression to genotype, must contain two columns, sample_id and participant_id, mapping IDs in the expression files to IDs in the genotype (these can be the same).
parameter: sample_participant_lookup = path()
# Options: gene, protein, atac
parameter: molecular_trait_type = "gene" 
# gtf annotation for RNA-seq data; other types of annotation index for protein and atac-seq
parameter: coordinate_annotation = path
# gene_id or gene_name for RNA-seq data, SOMAseqID for proteins for example
parameter: phenotype_id_column = "gene_id"
# Optional mapping file (protein_name_index for protein, etc.)
parameter: auxiliary_id_mapping = path()
# Optional processing flag
parameter: strip_id = False
parameter: sep = "\t"

input: phenoFile, coordinate_annotation
output: f'{cwd:a}/{_input[0]:bn}.bed.gz', f'{cwd:a}/{_input[0]:bn}.region_list.txt'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, tags = f'{step_name}_{_output[0]:bn}'
python: expand= "${ }", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout'

    import pandas as pd
    import numpy as np
    import qtl.io
    from pathlib import Path
    from collections import defaultdict
    import warnings

    # Function to convert gtf to bed
    def gtf_to_bed(annotation_gtf, feature='gene', exclude_chrs=[], phenotype_id='gene_id'):
        """
        Parse genes from GTF and return DataFrame with min start and max end positions.
        For each gene, uses the smallest position as start and largest as end position.
        """
        
        # Data collections
        gene_data = defaultdict(lambda: {'chr': '', 'start': float('inf'), 'end': 0, 'strand': ''})
        gene_names = {}  # To store gene_name for each gene_id
        
        if annotation_gtf.endswith('.gz'):
            opener = gzip.open(annotation_gtf, 'rt')
        else:
            opener = open(annotation_gtf, 'r')
            
        with opener as gtf:
            for row in gtf:
                row = row.strip().split('\t')
                if row[0][0] == '#' or row[2] != feature: 
                    continue  # Skip header or non-matching features
                
                # Parse attributes
                attributes = defaultdict()
                for a in row[8].replace('"', '').split(';')[:-1]:
                    kv = a.strip().split(' ')
                    if kv[0] != 'tag':
                        attributes[kv[0]] = kv[1]
                    else:
                        attributes.setdefault('tags', []).append(kv[1])
                
                # Get gene identifiers
                curr_gene_id = attributes['gene_id']
                curr_gene_name = attributes['gene_name']
                gene_names[curr_gene_id] = curr_gene_name
                
                # Update gene data
                data = gene_data[curr_gene_id]
                if data['chr'] == '':  # First entry for this gene
                    data['chr'] = row[0]
                    data['strand'] = row[6]
                
                # Update min start and max end
                start_pos = int(row[3]) - 1  # Convert to 0-based for BED
                end_pos = int(row[4]) - 1  # Both start and end are used
                data['start'] = min(data['start'], start_pos)
                data['end'] = max(data['end'], end_pos)
        
        # Convert to dataframe format
        chrom, start, end, ids, names, strand = [], [], [], [], [], []
        for gene_id, data in gene_data.items():
            chrom.append(data['chr'])
            start.append(data['start'])
            end.append(data['end'])
            ids.append(gene_id)
            names.append(gene_names[gene_id])
            strand.append(data['strand'])
        
        # Create DataFrame based on phenotype_id
        if phenotype_id == 'gene_id':
            bed_df = pd.DataFrame({
                'chr': chrom,
                'start': start,
                'end': end,
                'gene_id': ids,
                'strand': strand
            }, columns=['chr', 'start', 'end', 'gene_id', 'strand'], index=ids)
        elif phenotype_id == 'gene_name':
            bed_df = pd.DataFrame({
                'chr': chrom,
                'start': start,
                'end': end,
                'gene_id': names,
                'strand': strand
            }, columns=['chr', 'start', 'end', 'gene_id', 'strand'], index=names)
        
        # Filter out excluded chromosomes
        mask = np.ones(len(chrom), dtype=bool)
        for k in exclude_chrs:
            mask = mask & (bed_df['chr'] != k)
        bed_df = bed_df[mask]
        
        # Sort by chromosome and start position
        bed_df = bed_df.sort_values(['chr', 'start', 'end'])
        
        return bed_df

    def prepare_bed(df, bed_template_df, chr_subset=None):
        bed_df = pd.merge(bed_template_df, df, left_index=True, right_index=True)
        bed_df = bed_df.sort_values(['#chr', 'start', 'end'])
        if chr_subset is not None:
            bed_df = bed_df[bed_df.chr.isin(chr_subset)]
        return bed_df

    def load_and_preprocess_data(input_path, drop_columns):
        df = pd.read_csv(input_path, sep="${sep}", skiprows=0)
        dc = [col for col in df.columns if col in drop_columns] # Take intersection between df.columns and drop_columns
        df = df.drop(dc, axis=1) # drop the intersect
        if len(df.columns) < 2:
            raise ValueError("There are too few columns in the loaded dataframe, please check the delimiter of the input file. The default delimiter is tab")
        return df

    def rename_samples_using_lookup(df, lookup_path):
        lookup_file = Path(lookup_path)
        
        if not lookup_file.is_file():
            return df
            
        if lookup_file.suffix == '.csv':
            sep = ','
        elif lookup_file.suffix == '.tsv':
            sep = '\t'
        else:
            sep = '\t'  # Default to tab if unknown extension
            
        try:
            sample_participant_lookup = pd.read_csv(lookup_file, sep=sep, dtype={0:str, 1:str})
            if "genotype_id" in sample_participant_lookup.columns:
                sample_participant_lookup = sample_participant_lookup.set_index(sample_participant_lookup.columns[1])
                df.rename(columns=sample_participant_lookup.to_dict()["genotype_id"], inplace=True)
            else:
                # Fall back to the no header style
                sample_participant_lookup = pd.read_csv(lookup_file, sep=sep, header=None, dtype={0:str, 1:str})
                rename_dict = dict(zip(sample_participant_lookup[0], sample_participant_lookup[1]))
                df.rename(columns=rename_dict, inplace=True)
        except Exception as e:
            print(f"Warning: Error processing sample lookup file: {e}")
        return df

    def load_bed_template_from_gtf(input_path, phenotype_id_column):
        if sum(gtf_to_bed(input_path, feature='gene', phenotype_id="gene_id").index.duplicated()) > 0:
            raise ValueError(f"gtf file {input_path} needs to be collapsed into gene model by reference data processing module")

        bed_template_df_id = gtf_to_bed(input_path, feature='transcript', phenotype_id="gene_id")
        bed_template_df_name = gtf_to_bed(input_path, feature='transcript', phenotype_id="gene_name")
        bed_template_df = bed_template_df_id.merge(bed_template_df_name, on=["chr", "start", "end", "strand"])
        bed_template_df.columns = ["#chr", "start", "end", "gene_id", "strand", "gene_name"]
        bed_template_df = bed_template_df.set_index(phenotype_id_column, drop=False)
        return bed_template_df

    drop_cols = ["#chr", "chr", "start", "end", "stop", "annot.seqnames", "annot.start", "annot.end"]
    df = load_and_preprocess_data(${_input[0]:ar}, drop_cols)
    molecular_trait_type = "${molecular_trait_type}"
    df.set_index(df.columns[0], inplace=True)
    if ${True if sample_participant_lookup.is_file() else False}:
        df = rename_samples_using_lookup(df, "${sample_participant_lookup:a}")    
    # First column is always the phenotype ID column
    phenotype_id = df.columns.values[0]
    
    # ----- shared helpers / constants ---------------------------------------------------
    # Names of columns that are never sample-level phenotype data. Used by the ATAC branch
    # to derive sample columns by subtraction and by write_phenotype_outputs to guard the
    # analysis BED schema.
    ANNOTATION_COLUMNS = {"#chr", "start", "end", "ID", "strand", "gene_id", "gene_name"}

    def write_phenotype_outputs(merged_df, sample_cols, bed_path, meta_path):
        """Compose and write the two phenotype outputs from a single merged DataFrame.

        merged_df    : annotation columns + sample columns, must contain at minimum
                       [#chr, start, end, ID]; may also contain "strand".
        sample_cols  : explicit list of sample-level column names in merged_df.

        Analysis BED (bed_path)  : [#chr, start, end, ID, *sample_cols].
                                   Strand is intentionally omitted — tensorqtl and pecotmr
                                   expect exactly 4 metadata columns before samples.
        Sidecar TSV (meta_path)  : [#chr, start, end, ID] (+ "strand" if available) + path.
        """
        bed_df = merged_df[["#chr", "start", "end", "ID"] + sample_cols]
        meta_cols = ["#chr", "start", "end", "ID"]
        if "strand" in merged_df.columns:
            meta_cols.append("strand")
        meta_df = merged_df[meta_cols].copy()
        qtl.io.write_bed(bed_df, bed_path)
        meta_df.assign(path=bed_path).to_csv(meta_path, sep="\t", index=False)

    # ----- type-specific assembly --------------------------------------------------------
    # Each branch produces:
    #   merged_df   : annotation + samples (per write_phenotype_outputs contract)
    #   sample_cols : list of sample column names in merged_df
    if molecular_trait_type == "gene":
        if ${strip_id}:
            # Update the index by stripping the pattern "A | B" to "B"
            df.index = df.index.map(lambda x: x.split('|')[-1].strip() if '|' in x else x)

        bed_template_df = load_bed_template_from_gtf(${_input[1]:ar}, "${phenotype_id_column}")
        merged_df = prepare_bed(df, bed_template_df)
        merged_df = merged_df.drop_duplicates("gene_id", keep=False).rename(columns={'gene_id': 'ID'})
        sample_cols = list(df.columns)

    elif molecular_trait_type == "protein":
        auxiliary_mapping = Path("${auxiliary_id_mapping:a}")

        if auxiliary_mapping.is_file():
            # Use protein name mapping from auxiliary file
            df_info = pd.read_csv(auxiliary_mapping).rename(
                columns={phenotype_id: phenotype_id, 'EntrezGeneSymbol': 'gene_name'}
            )[['gene_name', phenotype_id, 'UniProt']]
            df = df_info.merge(df, on=phenotype_id).drop(phenotype_id, axis=1)
        else:
            # Extract UniProt ID from the phenotype ID (carried in the index) if no mapping file
            split_ids = df.index.astype(str).str.split('|', n=1, expand=True)
            df['UniProt'] = split_ids.get_level_values(1).values
            df.index = split_ids.get_level_values(0)
            df.index.name = "${phenotype_id_column}"

        bed_template_df = load_bed_template_from_gtf(${_input[1]:ar}, "${phenotype_id_column}")
        merged_df = prepare_bed(df, bed_template_df)
        merged_df["ID"] = merged_df["gene_id"] + "_" + merged_df["UniProt"]
        merged_df = merged_df.drop_duplicates("ID", keep=False)
        sample_cols = df.drop(["UniProt"], axis=1).columns.values.tolist()

    elif molecular_trait_type == "atac":
        # For ATAC, the coordinate_annotation is directly a BED file (no strand)
        bed_template_df = pd.read_csv("${_input[1]}", sep="\t").set_index("ID")
        bed_template_df = bed_template_df.assign(ID=bed_template_df.index)
        merged_df = prepare_bed(df, bed_template_df).drop_duplicates("ID", keep=False)
        sample_cols = [c for c in merged_df.columns if c not in ANNOTATION_COLUMNS]
    else:
        raise ValueError(f"Unsupported molecular_trait_type: {molecular_trait_type}")

    write_phenotype_outputs(merged_df, sample_cols, ${_output[0]:r}, ${_output[1]:r})

    # If annotating gene or protein, save a gene_list.tsv for gene partitioning
    if molecular_trait_type in ["gene", "protein"]:
        # Save the region list
        region_list = bed_template_df[bed_template_df.${phenotype_id_column}.isin(df.index)]
        region_list.to_csv('${cwd:a}/${_input[0]:bn}.gene_list.tsv', sep="\t", index=False)
    else:
        # For ATAC, we cannot partition it by gene.
        warnings.warn(f"Gene partitioning is not applicable for molecular trait type: {molecular_trait_type}. No gene_list.tsv will be generated.")
        


bash: expand= "$[ ]", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout', container = container
        stdout=$[_output[0]:n].stdout
        for i in $[_output[0]] ; do 
        echo "output_info: $i " >> $stdout;
        echo "output_size:" `ls -lh $i | cut -f 5  -d  " "`   >> $stdout;
        echo "output_rows:" `zcat $i | wc -l  | cut -f 1 -d " "`   >> $stdout;
        echo "output_headerow:" `zcat $i | grep "##" | wc -l `   >> $stdout;
        echo "output_column:" `zcat $i | grep -V "##" | head -1 | wc -w `   >> $stdout;
        echo "output_preview:"   >> $stdout;
        zcat $i  | grep -v "##" | head  | cut -f 1,2,3,4,5,6   >> $stdout ; done
        for i in $[_output[1]] ; do 
        echo "output_info: $i " >> $stdout;
        echo "output_size:" `ls -lh $i | cut -f 5  -d  " "`   >> $stdout;
        echo "output_rows:" `cat $i | wc -l  | cut -f 1 -d " "`   >> $stdout;
        echo "output_headerow:" `cat $i | grep "##" | wc -l `   >> $stdout;
        echo "output_column:" `cat $i | grep -V "##" | head -1 | wc -w `   >> $stdout;
        echo "output_preview:"   >> $stdout;
        cat $i  | grep -v "##" | head  | cut -f 1,2,3,4,5,6   >> $stdout ; done

## Implementation using biomaRt
This workflow adds the annotations of chr pos(TSS where start = end -1) and gene_ID to the `bed` file. **This workflow is obsolete**.

In [ ]:
[annotate_coord_biomart]
parameter: ensembl_version=int
input: phenoFile
output: f'{cwd:a}/{_input:bn}.bed.gz',
        f'{cwd:a}/{_input:bn}.region_list.txt'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output[0]:bn}'  
R:  expand= "$[ ]", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout' ,container = container

    library(biomaRt)
    library(dplyr)
    library(readr)

    # Clear biomart cache
    biomartCacheClear()

    # Read gene expression data
    gene_exp <- read_delim("$[_input[0]]", delim = "\t")

    # If column "#chr" exists, remove the first 3 columns (chr, start, and end)
    if("#chr" %in% colnames(gene_exp)){
      gene_exp <- gene_exp[, 4:ncol(gene_exp)]
    }

    # Connect to Ensembl biomart
    ensembl <- useEnsembl(biomart = "ensembl", dataset = "hsapiens_gene_ensembl", version = "$[ensembl_version]")

    # Get gene annotations from Ensembl
    ensembl_df <- getBM(attributes = c("ensembl_gene_id", "chromosome_name", "start_position", "end_position"), mart = ensembl)

    # Filter and rename columns
    my_genes_ann <- ensembl_df %>%
      filter(ensembl_gene_id %in% gene_exp$gene_ID, chromosome_name %in% 1:23) %>%
      rename("#chr" = chromosome_name, 
            "start" = start_position, 
            "end" = end_position, 
            "gene_ID" = ensembl_gene_id) %>%
      filter(gene_ID != "NA")

    # Save the annotations
    my_genes_ann %>%
      select(`#chr`, start, end, gene_ID) %>%
      write_delim(file = "$[_output[1]]", "\t")

    # Merge the annotations with the gene expression data, after modifying 'end' to be 'start + 1'
    my_gene_bed <- my_genes_ann %>%
      mutate(end = start + 1) %>%
      inner_join(gene_exp, by = "gene_ID") %>%
      arrange(`#chr`, start) %>%
    select(`#chr`, start, end, gene_ID, everything())

    # Save the final merged data
    write_tsv(my_gene_bed, file = "$[_output[0]:n]", na = "NA", append = FALSE, col_names = TRUE)


bash: expand = "$[ ]", stderr = f'{_output[0]}.stderr', stdout = f'{_output[0]}.stdout',container = container
    bgzip -f $[_output[0]:n]
    tabix -p bed $[_output[0]] -f

## Annotation of leafcutter isoform

The gtf used here should be the collapsed gtf, i.e. the final output of reference_data gtf processing and the one used to called rnaseq.

In [ ]:
[map_leafcutter_cluster_to_gene]
## Extract the code in case psichromatic needs to be processed the same way
## PheoFile in this step is the intron_count file
parameter: intron_count = path
# Defines the mapping strategy options: 'site' or 'region', with 'site' as the default. 
# The 'site' strategy maps introns to the start and end of each exon. 
# The 'region' strategy, to be recommended in leafcutter2, maps each intron based on it overlapping more than overlap_ratio of a gene's region.
parameter: map_stra = "site" 
# Define the overlap ratio as the proportion of the cluster length that intersects with a gene, used to determine mapping to the gene.
parameter: overlap_ratio = 0.8
parameter: coordinate_annotation = path
input: intron_count, coordinate_annotation
output: f'{cwd}/{_input[0]:b}.exon_list', f'{cwd}/{_input[0]:b}.leafcutter.clusters_to_genes.txt'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output[0]:bn}'  
python: expand= "${ }", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout', container = container
    import pandas as pd
    import qtl.annotation

    gtf = ${_input[1]:r}
    #there is no "gene_type" when using the uncollapsed gtf. Replace "gene_biotype" with "gene_type" in the gtf and temporarily save for use in the annotation
    has_gene_type = True
    with open(gtf, 'r') as file:
        for line in file:
            if line[0] == '#':
                continue

            row = line.strip().split('\t')
            chrom = row[0]
            # source = row[1]
            annot_type = row[2]

            if annot_type == "gene":
                if "gene_type" not in line:
                    has_gene_type = False
                break

    if has_gene_type == False:
        
        with open(gtf, 'r') as file:
            file_contents = file.read()
        updated_contents = file_contents.replace("gene_biotype", "gene_type")
        gtf = ${_input[1]:rn}.tmp${_input[1]:rx}
        with open(gtf, 'w') as file:
            file.write(updated_contents)


    # Load data
    #annot = qtl.annotation.Annotation(${_input[1]:r})
    annot = qtl.annotation.Annotation(gtf)
    exon_df = pd.DataFrame([[g.chr, e.start_pos, e.end_pos, g.strand, g.id, g.name]
                        for g in annot.genes for e in g.transcripts[0].exons],
                       columns=['chr', 'start', 'end', 'strand', 'gene_id', 'gene_name'])
    exon_df.to_csv(${_output[0]:r}, sep='\t', index=False)
    
    
R:expand= "${ }", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout', container = container
    suppressMessages(library(dplyr, quietly=TRUE))
    suppressMessages(library(stringr, quietly=TRUE))
    suppressMessages(library(foreach, quietly=TRUE))
    # leafcutter functions:
    
    #' Make a data.frame of meta data about the introns
    #' @param introns Names of the introns
    #' @return Data.frame with chr, start, end, cluster id
    #' @export
    get_intron_meta <- function(introns) {
      intron_meta <- do.call(rbind, strsplit(introns, ":"))  
      if (ncol(intron_meta) == 5) {
        colnames(intron_meta) <- c("chr", "start", "end", "clu", "category")
      } else {
        colnames(intron_meta) <- c("chr", "start", "end", "clu")
      }

      intron_meta <- as.data.frame(intron_meta, stringsAsFactors = FALSE)
      intron_meta$start <- as.numeric(intron_meta$start)
      intron_meta$end <- as.numeric(intron_meta$end)

      return(intron_meta)
    }
    #' Work out which gene each cluster belongs to. Note the chromosome names used in the two inputs must match.
    #' @param intron_meta Data frame describing the introns, usually from get_intron_meta
    #' @param exons_table Table of exons, see e.g. /data/gencode19_exons.txt.gz
    #' @return Data.frame with cluster ids and genes separated by commas
    #' @import dplyr
    #' @export
    map_clusters_to_genes_site <- function(intron_meta, exons_table) {
      suppressMessages(library(foreach, quietly=TRUE))
      gene_df <- foreach (chr=sort(unique(intron_meta$chr)), .combine=rbind) %dopar% {
    
        intron_chr <- intron_meta[ intron_meta$chr==chr, ]
        exons_chr <- exons_table[exons_table$chr==chr, ]
    
        exons_chr$temp <- exons_chr$start
        intron_chr$temp <- intron_chr$end
        three_prime_matches <- inner_join( intron_chr, exons_chr, by="temp")
    
        exons_chr$temp <- exons_chr$end
        intron_chr$temp <- intron_chr$start
        five_prime_matches <- inner_join( intron_chr, exons_chr, by="temp")
    
        all_matches <- rbind(three_prime_matches, five_prime_matches)[ , c("clu", "gene_name")]
    
        all_matches <- all_matches[!duplicated(all_matches),]
    
        if (nrow(all_matches)==0) return(NULL)
        all_matches$clu <- paste(chr,all_matches$clu,sep=':')
        all_matches
      }
    
      clu_df <- gene_df %>% group_by(clu) %>% summarize(genes=paste(gene_name, collapse = ","))
      class(clu_df) <- "data.frame"
      clu_df
    }
    
    map_clusters_to_genes_region <- function(intron_meta, exons_table, f) {
        #combine the exon position to get gene region
        merged_df <- exons_table %>%
            group_by(chr, gene_id) %>%
            summarize(start = min(start), end = max(end), .groups = 'drop') %>% as.data.frame()%>%
            .[,c("chr","start","end","gene_id")]
        # use bedtools to map the intron region to gene region 
         options(bedtools.path = "/home/rf2872/software/bedtools2/bin/") #Fixme: build bedtools in leafcutter sif
                map_res <- bedtoolsr::bt.intersect(a = intron_meta, b = merged_df,f = f, wa = TRUE, wb=TRUE)
        #organize output of mapped file
        exon_map <- map_res %>% 
            mutate(clu = paste(V1, V4, sep=":"), 
                   genes = .[[paste0("V", 4 + ncol(intron_meta))]]) %>%
                select(clu,genes)
          return(exon_map)
    }

    cat("LeafCutter: mapping clusters to genes\n")
    intron_counts <- read.table(${_input[0]:r}, header=TRUE, check.names=FALSE, row.names=4)
    intron_meta <- get_intron_meta(rownames(intron_counts))
    exon_table <- read.table(${_output[0]:r}, header=TRUE, stringsAsFactors=FALSE)
    if(!str_detect(intron_meta$chr[1],"chr")) {
        exon_table = exon_table %>% mutate(chr = str_remove_all(chr,"chr"))
    } else if (!any(str_detect(exon_table$chr[1],"chr"))) {
        exon_table = exon_table %>% mutate(chr = paste0("chr",chr))
    } else {exon_table = exon_table}
    stopifnot(is.element('gene_id', colnames(exon_table)))
    exon_table[, 'gene_name'] <- exon_table[, 'gene_id']
    if("${map_stra}" == "site"){
      cat("mapping clusters to genes by splicing site\n")
      m <- map_clusters_to_genes_site(intron_meta, exon_table)
      } else if("${map_stra}" == "region"){
      cat("mapping clusters to genes by overlapping gene region\n")
      m <- map_clusters_to_genes_region(intron_meta, exon_table, f = ${overlap_ratio})
    } else {
      stop("Map strategy should only be 'site' or 'region'.")
    }
    write.table(m, ${_output[1]:r}, sep = "\t", quote=FALSE, row.names=FALSE)
  
bash: expand= "$[ ]", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout', container = container
        stdout=$[_output[0]:n].stdout
        for i in $[_output] ; do 
        echo "output_info: $i " >> $stdout;
        echo "output_size:" `ls -lh $i | cut -f 5  -d  " "`   >> $stdout;
        echo "output_rows:" `zcat $i | wc -l  | cut -f 1 -d " "`   >> $stdout;
        echo "output_headerow:" `zcat $i | grep "##" | wc -l `   >> $stdout;
        echo "output_column:" `zcat $i | grep -V "##" | head -1 | wc -w `   >> $stdout;
        echo "output_preview:"   >> $stdout;
        zcat $i  | grep -v "##" | head  | cut -f 1,2,3,4,5,6   >> $stdout ; done

In [ ]:
[annotate_leafcutter_isoforms]
parameter: sample_participant_lookup = path()
parameter: coordinate_annotation = path
input: phenoFile, coordinate_annotation,output_from("map_leafcutter_cluster_to_gene")
output: f'{cwd:a}/{_input[0]:bn}.formated.bed.gz', f'{cwd:a}/{_input[0]:bn}.phenotype_group.txt'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output[0]:bn}'  
python: expand= "${ }", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout', container = container
    import pandas as pd
    import numpy as np
    import qtl.io
    import os
    from pathlib import Path

    # Load data
    if os.path.exists(${_input[1]:rn}.tmp${_input[1]:rx}):
        gtf = ${_input[1]:rn}.tmp${_input[1]:rx}
    else:
        gtf = ${_input[1]:r}
    #tss_df = qtl.io.gtf_to_tss_bed(${_input[1]:r})
    tss_df = qtl.io.gtf_to_tss_bed(gtf)
    bed_df = pd.read_csv(${_input[0]:ar}, sep='\t', skiprows=0)
    bed_df.columns.values[0] = "#chr" # Temporary
    sample_participant_lookup = Path("${sample_participant_lookup:a}")
    cluster2gene_dict = pd.read_csv(${_input[3]:r}, sep='\t', index_col=0).to_dict()
    cluster2gene_dict = cluster2gene_dict['genes']
    print('    ** assigning introns to gene mapping(s)')
    n = 0
    gene_bed_df = []
    group_s = {}
    for _,r in bed_df.iterrows():
        s = r['ID'].split(':')
        cluster_id = s[0]+':'+s[3]
        if cluster_id in cluster2gene_dict:
            gene_ids = cluster2gene_dict[cluster_id].split(',')
            for g in gene_ids:
                gi = r['ID']+':'+g
                gene_bed_df.append(tss_df.loc[g, ['chr', 'start', 'end']].tolist() + [gi] + r.iloc[4:].tolist())
                group_s[gi] = g
        else:
            n += 1
    if n > 0:
        print(f'    ** discarded {n} introns without a gene mapping')

    print('  * writing BED files for QTL mapping')
    gene_bed_df = pd.DataFrame(gene_bed_df, columns=bed_df.columns)
    # sort by TSS
    gene_bed_df = gene_bed_df.sort_values(['#chr', 'start'])
    #rename the samples if they named by file name (simply pick the first element with [.])
    gene_bed_df.columns = list(gene_bed_df.columns[:4]) + [name.split('.')[0] if 'junc' in name else name for name in gene_bed_df.columns[4:]]
    # change sample IDs to participant IDs
    if sample_participant_lookup.is_file():
        sample_participant_lookup_s = pd.read_csv(sample_participant_lookup, sep="\t", index_col=0, dtype={0:str,1:str})
        #gene_bed_df.rename(columns=sample_participant_lookup_s.to_dict(), inplace=True)
        # Create a dictionary mapping from sample_id to participant_id
        column_mapping = dict(zip(sample_participant_lookup_s.index, sample_participant_lookup_s['participant_id']))
        # Get the column names to be replaced
        column_names = gene_bed_df.columns[4:]
        # Replace the column names using the mapping dictionary
        #new_column_names = [column_mapping.get(col, 'missing_data') for col in column_names]
        #it should overlap with genotype in downstream anyways
        new_column_names = [column_mapping.get(col, col) for col in column_names]
        gene_bed_df.rename(columns=dict(zip(column_names, new_column_names)), inplace=True)

    gene_bed_df = gene_bed_df.drop_duplicates()
    qtl.io.write_bed(gene_bed_df, ${_output[0]:r})
    gene_bed_df[['start', 'end']] = gene_bed_df[['start', 'end']].astype(np.int32)
    gene_bed_df[gene_bed_df.columns[4:]] = gene_bed_df[gene_bed_df.columns[4:]].astype(np.float32)
    group_s_df =  pd.Series(group_s).sort_values().reset_index()
    group_s_df.columns = ['ID', 'gene']
    group_s_df.to_csv(${_output[1]:r}, sep='\t', index=False, header=True)

bash: expand= "$[ ]", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout', container = container
        stdout=$[_output[0]:n].stdout

        rm $[_input[1]:rn].tmp$[_input[1]:rx]

        for i in $[_output[0]] ; do 
        echo "output_info: $i " >> $stdout;
        echo "output_size:" `ls -lh $i | cut -f 5  -d  " "`   >> $stdout;
        echo "output_rows:" `zcat $i | wc -l  | cut -f 1 -d " "`   >> $stdout;
        echo "output_headerow:" `zcat $i | grep "##" | wc -l `   >> $stdout;
        echo "output_column:" `zcat $i | grep -V "##" | head -1 | wc -w `   >> $stdout;
        echo "output_preview:"   >> $stdout;
        zcat $i  | grep -v "##" | head  | cut -f 1,2,3,4,5,6   >> $stdout ; done
        for i in $[_output[1]] ; do 
        echo "output_info: $i " >> $stdout;
        echo "output_size:" `ls -lh $i | cut -f 5  -d  " "`   >> $stdout;
        echo "output_rows:" `cat $i | wc -l  | cut -f 1 -d " "`   >> $stdout;
        echo "output_headerow:" `cat $i | grep "##" | wc -l `   >> $stdout;
        echo "output_column:" `cat $i | grep -V "##" | head -1 | wc -w `   >> $stdout;
        echo "output_preview:"   >> $stdout;
        cat $i  | grep -v "##" | head  | cut -f 1,2,3,4,5,6   >> $stdout ; done

## Processing of psichomics output
It occurs that the psichomatic by default grouped the isoforms by gene name, so only thing needs to be done is to extract this information and potentially renamed the gene symbol into ENSG ID

## Anticipated Results

The pipeline produces output files in the `output/` subdirectory named after the workflow step. Verify success by checking that output files exist and are non-empty. See the **Output** section above for the expected file names and formats.

In [ ]:
[annotate_psichomics_isoforms]
parameter: sample_participant_lookup = path()
parameter: coordinate_annotation = path
input: phenoFile, coordinate_annotation
output: f'{cwd:a}/{_input[0]:bn}.formated.bed.gz', f'{cwd:a}/{_input[0]:bn}.phenotype_group.txt'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output[0]:bn}'  
python: expand= "${ }", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout', container = container
    import pandas as pd
    import numpy as np
    import qtl.io
    from pathlib import Path
    tss_df = qtl.io.gtf_to_tss_bed(${_input[1]:r}, feature='gene',phenotype_id = "gene_id" )
    bed_df = pd.read_csv(${_input[0]:ar}, sep='\t', skiprows=0)
    bed_df["gene_id"]  = [x[-1] for x in bed_df.ID.str.split("_")]
    sample_participant_lookup = Path("${sample_participant_lookup:a}")
    if "start" in  bed_df.columns:
        bed_df = bed_df.drop(["#Chr","start","end"],axis = 1)
    output = tss_df.merge(bed_df, left_on = ["gene_id"], right_on = ["gene_id"],how = "right").sort_values(["chr","start"])
    # change sample IDs to participant IDs
    if sample_participant_lookup.is_file():
        sample_participant_lookup_s = pd.read_csv(sample_participant_lookup, sep="\t", index_col=0, dtype={0:str,1:str})

        column_mapping = dict(zip(sample_participant_lookup_s.index, sample_participant_lookup_s['participant_id']))

        column_names = output.columns[4:]
        print(column_names)
        new_column_names = [column_mapping.get(col, col) for col in column_names]
        output.rename(columns=dict(zip(column_names, new_column_names)), inplace=True)

        #output.rename(columns=sample_participant_lookup_s.to_dict(), inplace=True)

    # Old code grouping by each gene
    #bed_output = output.drop("gene_id" , axis = 1)
    #phenotype_group = output[["ID","gene_id"]]

    bed_output = output.drop("gene_id" , axis = 1)
    bed_output = bed_output.rename(columns={bed_output.columns[0]: "#chr"})

    # R version 
    #output = output%>%mutate(gene_type_id = map_chr(str_split( ID ,"_"),~paste0(.x[length(.x)],"_",.x[1] )) )
    
    # corrected in python
    output = output.assign(gene_type_id = output['ID'].str.split('_').map(lambda x: x[-1] + '_' + x[0]))
    phenotype_group = output[["ID","gene_type_id"]]

    bed_output.to_csv(${_output[0]:nr},sep="\t",index = False)
    phenotype_group.to_csv(${_output[1]:r},sep="\t",index = False,header=False)

bash: expand= "${ }", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout', container = container
    bgzip -f ${_output[0]:n}
    tabix ${_output[0]}

bash: expand= "$[ ]", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout', container = container
        stdout=$[_output[0]:n].stdout
        for i in $[_output[0]] ; do 
        echo "output_info: $i " >> $stdout;
        echo "output_size:" `ls -lh $i | cut -f 5  -d  " "`   >> $stdout;
        echo "output_rows:" `zcat $i | wc -l  | cut -f 1 -d " "`   >> $stdout;
        echo "output_headerow:" `zcat $i | grep "##" | wc -l `   >> $stdout;
        echo "output_column:" `zcat $i | grep -V "##" | head -1 | wc -w `   >> $stdout;
        echo "output_preview:"   >> $stdout;
        zcat $i  | grep -v "##" | head  | cut -f 1,2,3,4,5,6   >> $stdout ; done
        for i in $[_output[1]] ; do 
        echo "output_info: $i " >> $stdout;
        echo "output_size:" `ls -lh $i | cut -f 5  -d  " "`   >> $stdout;
        echo "output_rows:" `cat $i | wc -l  | cut -f 1 -d " "`   >> $stdout;
        echo "output_headerow:" `cat $i | grep "##" | wc -l `   >> $stdout;
        echo "output_column:" `cat $i | grep -V "##" | head -1 | wc -w `   >> $stdout;
        echo "output_preview:"   >> $stdout;
        cat $i  | grep -v "##" | head  | cut -f 1,2,3,4,5,6   >> $stdout ; done